In [1]:
import os

import pandas as pd

# Get the absolute path of the script
current_dir = os.getcwd()

# Move up two levels to the project root
project_root = os.path.abspath(os.path.join(current_dir, "../.."))

# Construct the CSV path
csv_path = os.path.join(project_root, "data", "reviews_labeled_cleaned.csv")

# set current directory to project root
os.chdir(project_root)

In [2]:
df = pd.read_csv(csv_path)
df.head()

,source,size,comment,original,Satisfaction,Authenticity,Type,needs_response,Category,Impact,Language,Emotion,cleaned
0,Friedom,short,Not salty enough,Not salty enough,3,Legitimate,Complaint,Yes,Product Quality / Food Quality,Moderate,English,Disappointment,Not salty enough
1,Friedom,short,Overall good chicken and fries,Overall good chicken and fries,4,Legitimate,Praise,No,Product Quality / Food Quality,No issue,English,Satisfaction,Overall good chicken and fries
2,Friedom,medium,Over rated\nTiles that are sticky to each othe...,Over rated\nطوابل لازقين فبعضاهم و مزيتتين ميش...,1,Legitimate,Complaint,Yes,"Customer Service, Product Quality / Food Quali...",High,"English, Derja",Frustration,Over rated طوابل لازقين فبعضاهم و مزيتتين ميش ...
3,Friedom,short,Food took +30 minutes to come.,Food took +30 minutes to come.,2,Legitimate,Complaint,Yes,Delivery / Logistics,Low Impact,English,Disappointment,Food took 30 minutes to come.
4,Friedom,long,"●●●●Food : ● the buns are so soggy ""DEAD"" like...","●●●●Food : ● the buns are so soggy ""DEAD"" like...",2,Legitimate,Complaint,Yes,"Customer Service, Product Quality / Food Quali...",High,English,Anger,Food the buns are so soggy DEAD like they cant...


In [3]:
df["Type"].unique()

array(['Complaint', 'Praise', 'Suggestion', 'Question', 'General Comment'],
      dtype=object)

In [4]:
df["Emotion"].unique()

array(['Disappointment', 'Satisfaction', 'Frustration', 'Anger',
       'Neutral', 'Excitement', 'Gratitude', 'satisfaction'], dtype=object)

In [ ]:
from sklearn.preprocessing import LabelEncoder

from backend.app.services.cleaner import preprocess_comment

# Preprocess the comments
df["cleaned_comment"] = df["comment"].apply(preprocess_comment)
df["cleaned_original_comment"] = df["original"].apply(preprocess_comment)

# Encode target labels
df["Type_encoded"] = LabelEncoder().fit_transform(df["Type"])
df["Emotion_encoded"] = LabelEncoder().fit_transform(df["Emotion"])


## CLassical ML

In [ ]:
import numpy as np
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

spacy_model = spacy.load("en_core_web_md")
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

def tfidf_features(X):
    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words="english", sublinear_tf=True)
    return vectorizer.fit_transform(X)

def count_features(X):
    vectorizer = CountVectorizer(max_features=5000, ngram_range=(1,2), stop_words="english")
    return vectorizer.fit_transform(X)

def spacy_features(X):
    return np.array([spacy_model(text).vector for text in X])

def sbert_features(X):
    return sbert_model.encode(X, show_progress_bar=False)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(),
    "Naive Bayes": MultinomialNB(),  # Only works with sparse features like TF-IDF/Count
    "SVM": LinearSVC(),
}

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split


def benchmark_with_features(X_text, y, label_name, text_col_name, results_df):
    X_train, X_test, y_train, y_test = train_test_split(X_text, y, test_size=0.2, random_state=42)

    feature_extractors = {
        "TF-IDF": tfidf_features,
        "CountVectorizer": count_features,
        "Spacy": spacy_features,
        "SBERT": sbert_features,
    }

    for fe_name, fe_func in feature_extractors.items():
        try:
            print(f"\n🔍 Extracting features using: {fe_name}")
            X_train_vec = fe_func(X_train)
            X_test_vec = fe_func(X_test)

            for model_name, model in models.items():
                # Skip Naive Bayes for dense embeddings (Spacy, SBERT)
                if fe_name in ["Spacy", "SBERT"] and model_name == "Naive Bayes":
                    continue

                model.fit(X_train_vec, y_train)
                preds = model.predict(X_test_vec)

                acc = accuracy_score(y_test, preds)
                f1_macro = f1_score(y_test, preds, average="macro")
                f1_weighted = f1_score(y_test, preds, average="weighted")

                results_df.loc[len(results_df)] = {
                    "Model": model_name,
                    "Approach": fe_name,
                    "Language": "English" if text_col_name == "cleaned_comment" else "Original",
                    "Task": label_name,
                    "Accuracy": acc,
                    "F1_macro": f1_macro,
                    "F1_weighted": f1_weighted,
                }
        except Exception as e:
            print(f"⚠️ Skipping {fe_name} + {model_name} due to error: {e}")

    return results_df

In [ ]:
results_df = pd.DataFrame(columns=[
    "Model", "Approach", "Language", "Task",
    "Accuracy", "F1_macro", "F1_weighted",
])

# Type task
results_df = benchmark_with_features(df["cleaned_comment"], df["Type_encoded"], "Type", "cleaned_comment", results_df)
results_df = benchmark_with_features(df["cleaned_original_comment"], df["Type_encoded"], "Type", "cleaned_original_comment", results_df)

# Emotion task
results_df = benchmark_with_features(df["cleaned_comment"], df["Emotion_encoded"], "Emotion", "cleaned_comment", results_df)
results_df = benchmark_with_features(df["cleaned_original_comment"], df["Emotion_encoded"], "Emotion", "cleaned_original_comment", results_df)


In [17]:
results_df

,Model,Approach,Language,Task,Accuracy,F1_macro,F1_weighted
0,Logistic Regression,TF-IDF + Classical ML,English,Type,0.800995,0.405433,0.758877
1,Random Forest,TF-IDF + Classical ML,English,Type,0.791045,0.528760,0.763258
2,Naive Bayes,TF-IDF + Classical ML,English,Type,0.761194,0.368801,0.708093
3,SVM,TF-IDF + Classical ML,English,Type,0.840796,0.668993,0.826098
4,Logistic Regression,TF-IDF + Classical ML,English,Emotion,0.587065,0.183648,0.464776
5,Random Forest,TF-IDF + Classical ML,English,Emotion,0.552239,0.252520,0.484679
6,Naive Bayes,TF-IDF + Classical ML,English,Emotion,0.527363,0.121905,0.380995
7,SVM,TF-IDF + Classical ML,English,Emotion,0.601990,0.369172,0.561833
8,Logistic Regression,TF-IDF + Classical ML,Original,Type,0.736318,0.349524,0.681526
9,Random Forest,TF-IDF + Classical ML,Original,Type,0.711443,0.325742,0.651181


In [18]:
!pip install spacy sentence-transformers



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 1.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 KB 1.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.1/795.1 KB 1.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 KB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 1.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 KB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 1.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.8/204.8 KB 1.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.1/117.1 KB 1.8 MB/s eta 0:00:00a 0:00:01
  Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 KB 1.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.8/514.8 KB 1.8 MB/s e

In [19]:
!python3 -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 1.8 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


In [20]:
!pip install transformers datasets evaluate scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 KB 1.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 KB 1.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 KB 1.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 KB 1.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 KB 1.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 KB 1.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 1.8 MB/s eta 0:00:0000:0100:01
  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.5.1
    Uninstalling fsspec-2025.5.1:
      Succe

# Deep Learning 

In [ ]:
import evaluate
import pandas as pd
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

# Initialize result container if needed
results_df = pd.DataFrame(columns=["Model", "Approach", "Language", "Task", "Accuracy", "F1_macro", "F1_weighted"])

# Metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
        "f1_weighted": f1.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

def run_finetuning(df, model_name, text_col, label_col):
    le = LabelEncoder()
    df = df.copy()
    df = df[[text_col, label_col]].dropna()
    df["label"] = le.fit_transform(df[label_col])

    num_labels = len(le.classes_)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

    # Dataset
    dataset = Dataset.from_pandas(df[[text_col, "label"]].rename(columns={text_col: "text"}))
    dataset = dataset.train_test_split(test_size=0.2)

    tokenized_datasets = dataset.map(lambda x: tokenizer(x["text"], truncation=True, padding=True), batched=True)

    # Training setup
    training_args = TrainingArguments(
        output_dir="./tmp_output",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        evaluation_strategy="epoch",
        save_strategy="no",
        logging_dir="./logs",
        load_best_model_at_end=False,
        logging_steps=10,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    # Track results
    return {
        "Model": model_name,
        "Approach": "HF Transfer Learning",
        "Language": "English" if text_col == "cleaned_comment" else "Original",
        "Task": label_col,
        "Accuracy": metrics["eval_accuracy"],
        "F1_macro": metrics["eval_f1_macro"],
        "F1_weighted": metrics["eval_f1_weighted"],
    }


In [ ]:
# Define models per language
models_by_language = {
    "cleaned_comment": ["distilbert-base-uncased", "bert-base-uncased"],
    "cleaned_original_comment": ["bert-base-multilingual-cased", "xlm-roberta-base"],
}

# Define tasks
label_columns = ["Type", "Emotion"]

# Run loop
for text_col, model_list in models_by_language.items():
    for model_name in model_list:
        for label_col in label_columns:
            print(f"\nTraining {model_name} on {text_col} to classify {label_col}...")
            try:
                result = run_finetuning(df, model_name, text_col, label_col)
                results_df.loc[len(results_df)] = result
                print(f"✅ Done: {result}")
            except Exception as e:
                print(f"❌ Failed: {model_name} - {text_col} - {label_col} due to {e}")